# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Support Vector Machines (SVM)

Find the optimal decision boundary between classes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

## Step 2: How SVM Works

**Goal:** Find the hyperplane that separates classes with the widest possible margin.

```
Good boundary: wide margin → generalizes well

     ■ ■
  ■  ■  ■     ─ margin
     ■  | ●●●
        |● ●●●
        |   ●●●
    (boundary)

Bad boundary: narrow margin → likely to misclassify

     ■ ■       ─ narrow margin
  ■  ■  ■  ────┤───●●●
     ■  | ● ●●●
        |   ●●●
```

In [ ]:
# Linear SVM
svm = SVC(kernel='linear', C=1.0, random_state=42)
svm.fit(X_train_scaled, y_train)

y_pred = svm.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, svm.decision_function(X_test_scaled))

print(f"SVM (linear) - Accuracy: {accuracy:.4f}, AUC: {auc:.4f}")
print(f"\nNumber of support vectors: {len(svm.support_)}")
print(f"(Out of {len(X_train)} training examples)")

## Step 3: The C Parameter (Regularization)

- **Small C:** Allows margin violations → wider margin, higher bias, lower variance
- **Large C:** Penalizes violations → narrower margin, lower bias, higher variance (overfitting)

In [ ]:
# Effect of C
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
train_scores = []
test_scores = []

for C in C_values:
    svm = SVC(kernel='linear', C=C, random_state=42)
    svm.fit(X_train_scaled, y_train)
    
    train_score = svm.score(X_train_scaled, y_train)
    test_score = svm.score(X_test_scaled, y_test)
    
    train_scores.append(train_score)
    test_scores.append(test_score)
    print(f"C={C:>6}: Train={train_score:.4f}, Test={test_score:.4f}")

## Step 4: Non-Linear SVM (RBF Kernel)

Kernel trick: map to high-dimensional space without explicit computation.

In [ ]:
# RBF (Radial Basis Function) kernel - non-linear
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)

y_pred_rbf = svm_rbf.predict(X_test_scaled)
accuracy_rbf = accuracy_score(y_test, y_pred_rbf)
auc_rbf = roc_auc_score(y_test, svm_rbf.decision_function(X_test_scaled))

print(f"SVM (RBF) - Accuracy: {accuracy_rbf:.4f}, AUC: {auc_rbf:.4f}")
print(f"Number of support vectors: {len(svm_rbf.support_)}")

## Step 5: Gamma Parameter (RBF Only)

- **Low gamma:** Each training point has wide influence → smoother boundary
- **High gamma:** Each point has narrow influence → more complex boundary

In [ ]:
# Effect of gamma
gamma_values = [0.001, 0.01, 0.1, 1, 10]

print(f"{'Gamma':<8} {'Train':>8} {'Test':>8}")
print("-" * 25)

for gamma in gamma_values:
    svm = SVC(kernel='rbf', C=1.0, gamma=gamma, random_state=42)
    svm.fit(X_train_scaled, y_train)
    
    train_score = svm.score(X_train_scaled, y_train)
    test_score = svm.score(X_test_scaled, y_test)
    print(f"{gamma:<8} {train_score:>8.4f} {test_score:>8.4f}")

## Step 6: Model Comparison

In [ ]:
# Compare kernels
print("\n" + "="*50)
print("KERNEL COMPARISON")
print("="*50)

kernels_to_try = [
    ('linear', {}),
    ('rbf', {'gamma': 'scale'}),
    ('poly', {'degree': 3})
]

for kernel, params in kernels_to_try:
    svm = SVC(kernel=kernel, C=1.0, random_state=42, **params)
    svm.fit(X_train_scaled, y_train)
    
    train_score = svm.score(X_train_scaled, y_train)
    test_score = svm.score(X_test_scaled, y_test)
    auc = roc_auc_score(y_test, svm.decision_function(X_test_scaled))
    
    print(f"\n{kernel.upper()}")
    print(f"  Accuracy: {test_score:.4f}")
    print(f"  AUC:      {auc:.4f}")